# feeding / exchanger 릴레이 진단

두 액추에이터를 **따로** 진단한다. 판단 근거가 서로 다르기 때문이다.

| | 관찰 대상 | 왜 |
|---|---|---|
| **feeding** | **수위 상승** | 온도로는 8개 중 4개 탱크만 잡혔다. 수위로 보면 8개 전부 잡힌다 (Cohen's d 2.5~13.9). 열 지연도 제어 역인과도 없는 직접 관측 |
| **exchanger** | **온도 변화** | 수위와 무관한 액추에이터. 온도 기반 검출비가 6/8 탱크에서 1.0 이상 |

## 핵심 아이디어: ON/OFF 비교가 아니라 **이벤트끼리** 비교한다

exchanger 는 "온도가 높을 때 켜진다"는 선택 편향이 있어서, ON 구간과 OFF 구간을
비교하면 그 편향이 그대로 섞여 들어온다. 그래서 앞서 매칭 비교가 필요했다.

그런데 **고장 감지 목적에서는 그 문제가 사라진다.** "이번 exchanger 동작이
지금까지의 exchanger 동작들과 같은가?" 를 물으면, 선택 편향은 모든 이벤트에
똑같이 걸려 있으므로 서로 상쇄된다. feeding 도 마찬가지다.

그래서 두 진단 모두 같은 틀을 쓴다.

1. 이벤트(연속 ON 구간) 검출
2. 이벤트마다 반응 지표 측정 — feeding 은 수위 상승률, exchanger 는 온도 기울기
3. 과거 이벤트 분포로 기준선(중앙값·MAD) 구성
4. **로버스트 z-score** — 평균/표준편차 대신 중앙값/MAD 를 쓴다. 고장 이벤트가
   기준선 자체를 오염시키는 걸 막기 위해서다
5. **CUSUM** 으로 지속성 판정 — 한 번 튄 건 노이즈일 수 있지만 계속 한쪽으로
   쌓이면 고장이다. 경고/확정 두 단계 임계값으로 "애매함" 상태를 만든다

## 수위는 고해상도 실수값(`level_f`)을 쓴다

`TK_Level_PV` 는 정수라 ±0.5 오차가 실린다. 센서 실제 잡음은 0.04~0.08 이므로
정수화가 잡음보다 10배 큰 오차를 넣고 있었다. 원시 아날로그에서 다시 계산한
`level_f` 는 해상도가 38~67배 높다.

## 0. 설정

In [ ]:
import glob
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.getcwd())
from Tank_Temp_Factor_Analysis import TANKS, WAGON_TAG

import matplotlib
import matplotlib.pyplot as plt
# ↑ 위 import 가 백엔드를 Agg 로 바꾸므로 inline 은 그 뒤에
%matplotlib inline
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

SAVE_DIR = "extracted_csv"

GRID_SEC = 5
EXCLUDE_HOURS = (0, 7)
RUNNING_WINDOW_MIN = 30

TAIL_SEC = 60          # 이벤트 종료 후 반응을 더 볼 시간 (열/유입 지연)

#: 이보다 짧은 이벤트는 측정이 부정확해 제외. **10초로 잡으면 안 된다** —
#: I2/I3 의 feeding 펄스는 중앙값 7.3초라 통째로 날아간다 (I3: 265개 -> 4개).
MIN_EVENT_SEC = 5

#: 기준선을 **직전 N개 이벤트**로 잡는다 (고정 구간이 아니라 이동 기준선).
#: 4.7일 전체를 고정 기준선으로 쓰면 정상적인 공정 변동까지 CUSUM 에 쌓여서
#: 멀쩡한 탱크가 전부 "확정"으로 나온다. 릴레이 고장은 서서히 나빠지는 게
#: 아니라 한순간에 끊기는 **계단형 변화**라, 직전 이벤트들과 비교해도 안 놓친다.
BASELINE_WINDOW = 30

CUSUM_K = 0.5          # 여유분 (이만큼은 정상 변동으로 본다)

#: z 를 이 값으로 자른다. 이 데이터는 꼬리가 두꺼워서 (라인이 멈추다 만 이벤트 등)
#: 가끔 z=-10 짜리가 꽂히는데, 자르지 않으면 그 하나로 CUSUM 이 폭발한다.
#: 실측: P1 feeding 정상 데이터의 CUSUM 최대값이 238 -> 23 으로 떨어진다.
Z_CLIP = 4.0

#: **임계값은 교과서 기본값(5)을 쓰면 안 된다.** 정상 데이터에서도 CUSUM 최대값이
#: 탱크마다 4~57 로 제각각이라, 5 로 두면 8개 탱크가 전부 "확정 고장"으로 나온다.
#: 대신 정상 구간에서 실제로 관측된 CUSUM 최대값에 여유를 곱해서 탱크마다 따로 잡는다.
THRESH_MARGIN = 1.3

## 1. 데이터 로드

라인 정지 구간(9/13 종일 등), 00~07시, 알람 구간은 제외한다.
이벤트 검출에는 원본 시각 해상도가 필요하므로 리샘플하지 않고 그대로 쓴다.

In [ ]:
_CACHE = {}


def load(tank_key):
    if tank_key in _CACHE:
        return _CACHE[tank_key]

    cfg = TANKS[tank_key]
    path = sorted(glob.glob(os.path.join(SAVE_DIR, f"{tank_key}_Temp_vs_Factors_*.csv")))[-1]
    use = ["Time", "source", WAGON_TAG, cfg["temp"], "level_f",
           cfg["exchanger"], cfg["feeding"], cfg["heat"]]
    raw = pd.read_csv(path, index_col="Time", parse_dates=True, usecols=use).sort_index()

    d = pd.DataFrame({
        "T": raw[cfg["temp"]], "level": raw["level_f"],
        "exch": raw[cfg["exchanger"]], "feed": raw[cfg["feeding"]],
        "heat": raw[cfg["heat"]], "wagon": raw[WAGON_TAG],
        "diag": (raw["source"] == "DIAGNOSTIC").astype(float),
    })

    # 라인 가동 여부 — wagon 이 최근 30분 안에 움직였는가
    w = d["wagon"].resample("1min").last().ffill()
    run_min = (w.diff().abs() > 0).rolling(RUNNING_WINDOW_MIN, min_periods=1).max()
    d["running"] = run_min.reindex(d.index, method="ffill").fillna(0)

    h = d.index.hour
    d["ok"] = (((h < EXCLUDE_HOURS[0]) | (h >= EXCLUDE_HOURS[1]))
               & (d["diag"] == 0) & (d["running"] == 1))

    _CACHE[tank_key] = d
    print(f"[{tank_key}] {len(d):,}행  |  유효 {int(d['ok'].sum()):,}행  |  "
          f"{d.index.min()} ~ {d.index.max()}")
    return d


for tk in TANKS:
    load(tk)

## 2. 공통 도구 — 이벤트 검출, 로버스트 z-score, CUSUM

In [ ]:
def events(d, col):
    """연속 ON 구간을 (시작, 끝) 으로 묶는다. 유효 구간 안에 온전히 들어가는 것만."""
    s = (d[col] == 1) & d["ok"]
    start = s & ~s.shift(1, fill_value=False)
    grp = start.cumsum()[s]
    out = []
    for _, idx in d.index[s].to_series().groupby(grp):
        t0, t1 = idx.iloc[0], idx.iloc[-1]
        if (t1 - t0).total_seconds() >= MIN_EVENT_SEC:
            out.append((t0, t1))
    return out


def _mad_scale(base):
    med = np.median(base)
    mad = np.median(np.abs(base - med))
    scale = 1.4826 * mad          # 정규분포에서 표준편차와 맞도록 보정
    if scale < 1e-9:
        scale = np.std(base) if np.std(base) > 1e-9 else 1.0
    return med, scale


def rolling_z(values, window=BASELINE_WINDOW, clip=Z_CLIP):
    """직전 `window` 개 이벤트를 기준선으로 삼은 로버스트 z-score.

    중앙값/MAD 를 쓰는 이유: 고장 이벤트 몇 개가 기준선 자체를 끌어당기는 걸 막는다.
    `clip` 으로 자르는 이유는 위 Z_CLIP 설명 참고.
    """
    z = np.full(len(values), np.nan)
    for i in range(window, len(values)):
        med, scale = _mad_scale(values[i - window:i])
        z[i] = (values[i] - med) / scale
    return np.clip(z, -clip, clip) if clip else z


def cusum(z, k=CUSUM_K):
    """하향 CUSUM — 반응이 기준보다 계속 작게 나오면 쌓인다.

    **부호 정규화가 선행되어야 한다.** feeding 은 기준 반응이 양수(수위 상승)지만
    exchanger 는 음수(온도 하강)다. 그대로 두면 exchanger 의 "릴레이 사망"(기울기가
    0 으로 접근)이 위쪽 방향이라 하향 CUSUM 에 안 걸린다. 아래 `diagnose` 에서
    기준 중앙값의 부호를 곱해 "반응이 클수록 큰 값"으로 맞춘 뒤 넣는다.
    """
    s, out = 0.0, []
    for v in z:
        s = max(0.0, s - v - k)
        out.append(s)
    return np.array(out)


def verdict(cs, thr):
    """3단계 판정 — 임계값의 60% 를 넘으면 "애매"로 먼저 알린다."""
    if cs >= thr:
        return "확정"
    if cs >= thr * 0.6:
        return "애매"
    return "정상"

## 3. feeding 진단 — 수위 상승률

이벤트마다 **수위 상승 속도**(단위/초)를 잰다. 밸브가 정상이면 유량이 일정하므로
이 값이 이벤트마다 비슷해야 한다. 릴레이가 죽으면 0 에 가까워진다.

이벤트 종료 후 `TAIL_SEC` 만큼 더 보는 이유: 수위 반응에 지연이 있다 (I3 기준 +25초).

In [ ]:
def feed_metrics(tank_key):
    d = load(tank_key)
    rows = []
    for t0, t1 in events(d, "feed"):
        pre = d["level"].asof(t0 - pd.Timedelta(seconds=5))
        post = d["level"].asof(t1 + pd.Timedelta(seconds=TAIL_SEC))
        dur = (t1 - t0).total_seconds()
        if pd.isna(pre) or pd.isna(post) or dur <= 0:
            continue
        rows.append({"start": t0, "dur_sec": round(dur, 1),
                     "수위상승": round(post - pre, 3),
                     "상승률": (post - pre) / dur})
    return pd.DataFrame(rows)


FEED = {tk: feed_metrics(tk) for tk in TANKS}
pd.DataFrame([{"탱크": tk, "이벤트수": len(v),
               "상승률 중앙값": round(v["상승률"].median(), 4) if len(v) else None,
               "수위상승 중앙값": round(v["수위상승"].median(), 3) if len(v) else None}
              for tk, v in FEED.items()])

In [ ]:
def diagnose(df, metric):
    """이동 기준선으로 z-score 를 내고 CUSUM 으로 지속성을 본다."""
    if len(df) < BASELINE_WINDOW + 10:
        return None, {"이벤트수": len(df), "기준_중앙값": None, "감시이벤트": 0,
                      "정상CUSUM최대": None, "산정임계값": None}

    vals = df[metric].values.astype(float)
    sign = np.sign(np.median(vals))          # 반응 방향 정규화 (위 cusum 설명 참고)
    z = rolling_z(vals * sign)
    ok = ~np.isnan(z)
    cs = cusum(z[ok])

    out = df.copy()
    out["z"] = z
    out["cusum"] = np.nan
    out.loc[out.index[ok], "cusum"] = cs

    # 이 구간을 정상으로 보고 임계값을 잡는다 (일별 중앙값이 거의 고정인 걸 확인했다)
    thr = max(float(cs.max()) * THRESH_MARGIN, 5.0) if len(cs) else 5.0

    return out, {"이벤트수": len(df), "기준_중앙값": round(float(np.median(vals)), 4),
                 "감시이벤트": int(ok.sum()),
                 "정상CUSUM최대": round(float(cs.max()), 1) if len(cs) else None,
                 "산정임계값": round(thr, 1)}


rows = []
FEED_DIAG = {}
for tk in TANKS:
    res, info = diagnose(FEED[tk], "상승률")
    FEED_DIAG[tk] = res
    rows.append({"탱크": tk, **info})
pd.DataFrame(rows)

## 4. exchanger 진단 — 온도 기울기 (⚠️ 이 방법은 실패했다)

이벤트마다 **온도 변화 기울기**(℃/분)를 잰다. 선택 편향(온도가 높을 때 켜짐)은
모든 이벤트에 똑같이 걸리므로 이벤트끼리 비교하면 상쇄된다 — 여기까지는 맞다.

**그런데 검증해보니 고장을 못 잡는다.** 아래 주입 테스트에서 exchanger 를
강제로 죽여도 8개 중 6개 탱크에서 CUSUM 이 임계값에 도달하지 못한다.
이벤트마다 조건(시작 온도, heat 상태, 지속시간)이 달라서 기울기 자체의
편차가 효과보다 크기 때문이다.

조건을 통제한 방법도 시도했지만 마찬가지였다 — exchanger 를 뺀 ARX 로 온도를
설명하고 남은 잔차를 exchanger 기여분으로 보는 방식인데, Cohen's d 가
0.11~0.64 에 그쳤다. (앞서 ARX 계수로 추정했던 검출비 1.0~2.5 는 **자기상관을
무시한 과대평가**였다. 표본이 40만 개처럼 보여도 독립 정보는 이벤트 수만큼뿐이다.)

`Scale_Out___TT_EX_*` 라는 exchanger 전용 온도 센서 태그도 찾았지만 **값이 전부
0 인 미사용 채널**이었다. feeding 의 수위 같은 직접 신호가 exchanger 에는 없다.

그래서 아래 코드는 **남겨두되 결과를 믿으면 안 된다.** exchanger 를 제대로
진단하려면 유량계나 밸브 개도 피드백 같은 별도 계측이 필요하다.

In [ ]:
def exch_metrics(tank_key):
    d = load(tank_key)
    rows = []
    for t0, t1 in events(d, "exch"):
        seg = d.loc[t0: t1 + pd.Timedelta(seconds=TAIL_SEC), "T"].dropna()
        if len(seg) < 5:
            continue
        x = (seg.index - seg.index[0]).total_seconds().values
        if x[-1] <= 0:
            continue
        slope = np.polyfit(x, seg.values, 1)[0] * 60.0     # ℃/분
        rows.append({"start": t0, "dur_sec": round((t1 - t0).total_seconds(), 1),
                     "시작온도": round(seg.iloc[0], 1),
                     "기울기": slope})
    return pd.DataFrame(rows)


EXCH = {tk: exch_metrics(tk) for tk in TANKS}

rows = []
EXCH_DIAG = {}
for tk in TANKS:
    res, info = diagnose(EXCH[tk], "기울기")
    EXCH_DIAG[tk] = res
    rows.append({"탱크": tk, **info})
pd.DataFrame(rows)

## 5. 종합 판정

In [ ]:
rows = []
for tk in TANKS:
    r = {"탱크": tk}
    for name, diag, src in [("feeding", FEED_DIAG, FEED), ("exchanger", EXCH_DIAG, EXCH)]:
        d = diag[tk]
        r[f"{name}_이벤트"] = len(src[tk])
        if d is None:
            r[f"{name}_판정"] = "표본부족"
        else:
            cs = d["cusum"].dropna()
            thr = max(cs.max() * THRESH_MARGIN, 5.0) if len(cs) else 5.0
            r[f"{name}_현재CUSUM"] = round(float(cs.iloc[-1]), 2) if len(cs) else None
            r[f"{name}_임계값"] = round(thr, 1)
            r[f"{name}_판정"] = verdict(cs.iloc[-1] if len(cs) else 0, thr)
    rows.append(r)
pd.DataFrame(rows)

## 6. 탱크별 상세 보기

`TANK` 를 바꿔가며 실행하세요. 위: 이벤트별 지표와 기준선, 아래: CUSUM 누적.

In [ ]:
TANK = "I3"

fig, axes = plt.subplots(2, 2, figsize=(15, 7), sharex="col")
for j, (name, diag, metric) in enumerate([("feeding", FEED_DIAG, "상승률"),
                                          ("exchanger", EXCH_DIAG, "기울기")]):
    d = diag[TANK]
    if d is None:
        axes[0][j].set_title(f"{TANK} {name} — 표본 부족")
        continue
    axes[0][j].plot(d["start"], d[metric], ".", markersize=4, color="#1f77b4")
    axes[0][j].axhline(np.median(d[metric].values), color="#d62728",
                       linestyle="--", linewidth=1, label="전체 중앙값")
    axes[0][j].set_title(f"{TANK} — {name}: 이벤트별 {metric}")
    axes[0][j].legend(fontsize=8)
    axes[0][j].grid(alpha=0.3)

    axes[1][j].plot(d["start"], d["cusum"], color="#2ca02c", linewidth=1.2)
    _cs = d["cusum"].dropna()
    _thr = max(_cs.max() * THRESH_MARGIN, 5.0) if len(_cs) else 5.0
    axes[1][j].axhline(_thr * 0.6, color="orange", linestyle="--", linewidth=1, label="경고")
    axes[1][j].axhline(_thr, color="red", linestyle="--", linewidth=1, label="확정")
    axes[1][j].set_title(f"{name}: 하향 CUSUM")
    axes[1][j].legend(fontsize=8)
    axes[1][j].grid(alpha=0.3)

fig.autofmt_xdate()
fig.tight_layout()

## 7. 민감도 확인 — 가짜 고장을 주입하면 잡히는가

실제 고장 사례가 데이터에 없으므로, **마지막 N개 이벤트의 반응을 강제로 0 으로**
바꿔 넣고 CUSUM 이 몇 개만에 확정 알람에 도달하는지 본다.
이게 "릴레이가 죽으면 몇 번 만에 알아채는가" 에 대한 답이다.

In [ ]:
def injection_test(df, metric, n_inject=20):
    """마지막 n_inject 개 이벤트의 반응을 0 으로 만들고 몇 번째에 잡히는지 본다."""
    if len(df) < BASELINE_WINDOW + 10:
        return None
    vals = df[metric].values.astype(float).copy()
    sign = np.sign(np.median(vals))
    vals[-n_inject:] = 0.0                    # 릴레이 사망 = 반응 없음
    # 정상 구간으로 임계값을 먼저 잡는다
    z0 = rolling_z(df[metric].values.astype(float) * sign)
    cs0 = cusum(z0[~np.isnan(z0)])
    thr = max(float(cs0.max()) * THRESH_MARGIN, 5.0)

    z = rolling_z(vals * sign)
    cs = cusum(z[~np.isnan(z)])
    tail = cs[-n_inject:]
    hit = int(np.argmax(tail >= thr)) + 1 if (tail >= thr).any() else None
    warn = int(np.argmax(tail >= thr * 0.6)) + 1 if (tail >= thr * 0.6).any() else None
    return {"임계값": round(thr, 1), "경고까지": warn, "확정까지": hit}


rows = []
for tk in TANKS:
    r = {"탱크": tk}
    for name, src, metric in [("feeding", FEED, "상승률"), ("exchanger", EXCH, "기울기")]:
        t = injection_test(src[tk], metric)
        r[f"{name}_임계값"] = t["임계값"] if t else None
        r[f"{name}_경고"] = t["경고까지"] if t else None
        r[f"{name}_확정"] = t["확정까지"] if t else None
    rows.append(r)
print("릴레이가 죽었을 때 몇 번째 이벤트에서 잡히는가 (None = 못 잡음)")
pd.DataFrame(rows)